# finetune: seed별 순차 실행
02는 seed별 ArcFace 부모, 03은 같은 부모에서 세 head를 순차 학습합니다.

## 1. 설정
공통 변경은 00, 이 단계만 변경하려면 아래 변수를 편집하세요.

In [ ]:
from pathlib import Path
import sys, json
BASE_DIR = Path("F:/FaceX-Zoo")
SETTINGS_PATH = BASE_DIR / "fine-tune/data/notebook_settings.json"
for path in (BASE_DIR, BASE_DIR / "fine-tune"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
from modules.notebook_workflow import resolve_settings, save_settings, plan_matrix
# 00의 공통값을 읽습니다. 아래 변수 변경이 저장값/YAML보다 우선합니다.
saved = resolve_settings(BASE_DIR, SETTINGS_PATH, require_saved=True)
SEEDS = list(saved["project"]["seed_list"])  # 예: [8972]. 이후 단계도 같은 목록을 사용하세요.
HEADS = list(saved["campaign"]["heads"])  # 기본 세 head. 02는 ArcFace 부모만 생성합니다.
KEEP_RAW_RESULTS = saved["project"]["keep_raw_results"]
NUM_WORKERS = 0
DEVICE = saved["project"]["device"]
DATASET = dict(saved["dataset"])  # 경로·input_size·pixel_mean/std
BACKBONE = dict(saved["backbone"])
TRAINING = dict(saved["training"])  # epoch·batch·학습률 등
HEAD_PARAMETERS = saved["heads"]
PATHS = dict(saved["paths"])
REUSE_COMPLETED = saved["campaign"]["reuse_completed"]
EXECUTE = False  # True일 때만 목록 생성/학습/export를 수행합니다.
STAGE = "finetune"  # 변경하지 마세요
RUN_SMOKE_TEST = False  # True는 더미 1-step, 정식 모델 아님
RESUME_CHECKPOINTS = {}  # 예: {"finetune_adaface_seed8972": "F:/.../epoch_5.pt"}
# 현재 노트북 변수 -> 실제 모듈 설정. 입력 112x112 이외는 현 구조에서 명시적 중단.
DATASET.update(train_archive=str(Path(DATASET["raw_root"])/"data/vggface2_train.tar.gz"),
               test_archive=str(Path(DATASET["raw_root"])/"data/vggface2_test.tar.gz"),
               meta_csv=str(Path(DATASET["raw_root"])/"meta/identity_meta.csv"))
settings = resolve_settings(BASE_DIR, SETTINGS_PATH, overrides={
    "project": {"seed": SEEDS[0], "seed_list": SEEDS, "keep_raw_results": KEEP_RAW_RESULTS, "device": DEVICE},
    "campaign": {"heads": HEADS, "reuse_completed": REUSE_COMPLETED},
    "dataset": DATASET, "backbone": BACKBONE, "heads": HEAD_PARAMETERS,
    "training": {**TRAINING, "num_workers": NUM_WORKERS}, "paths": PATHS,
})


## 2. 계획

In [ ]:
plan = plan_matrix(settings, STAGE)
for row in plan:
    print(row)
print("예정:",len(plan),"EXECUTE:",EXECUTE,"SMOKE:",RUN_SMOKE_TEST)

## 3. 재개 정책

In [ ]:
print("재개:",RESUME_CHECKPOINTS)
print("완료 run은 설정/hash 확인 후 재사용. 미완료 run은 재개 경로를 지정하세요.")
print("학습 중 공유 목록을 다시 생성하지 마세요.")

## 4. 순차 실행
정식 실행은 CUDA 필수입니다.

In [ ]:
from modules.notebook_workflow import run_training_matrix
results = []
if EXECUTE:
    results = run_training_matrix(settings,STAGE,RESUME_CHECKPOINTS,smoke=RUN_SMOKE_TEST)
else:
    print("계획 모드: 학습 미실행. 첫 셀 EXECUTE=True 후 순서대로 실행하세요.")

## 5. 완료 상태

In [ ]:
for result in results:
    print(result)
print("정식 완료:",sum(r.get("completed") is True for r in results),"/",len(plan))
print("실패 시 summary에 미실행 범위를 기록하고 후속 작업을 중단합니다.")
print("해석 안내:",BASE_DIR/"fine-tune/README.md")